In [1]:
# TASK 1: ImpulseBlock — Inhibitory Control
# CogExec-EF Executive Functions Benchmark
#
# Cognitive faculty : Executive Functions → Inhibitory Control
# Research basis    : Cognitive Reflection Test (Frederick, 2005) extended
#                     to logic, language, riddles, and lateral thinking
# What it isolates  : Can the model suppress a highly tempting but WRONG
#                     answer (System 1 intuition) and produce the correct
#                     one (System 2 reasoning)?
# Scoring           : tuple[int, int] → (passes, 2) — partial credit
#                     1 pt: correct answer present
#                     1 pt: tempting wrong answer absent

import kaggle_benchmarks as kbench
import pandas as pd
import re

df = pd.read_csv("/kaggle/input/datasets/jaytalwar2005/cogexec-ef-benchmark-data/impulseblock_150_FINAL_v3.csv")
print(f"Loaded {len(df)} rows")
print(f"Difficulty:    {df['difficulty'].value_counts().to_dict()}")
print(f"Category:      {df['category'].value_counts().to_dict()}")
print(f"EF Component:  {df['ef_component'].value_counts().to_dict()}")
print(f"Scoring Method:{df['scoring_method'].value_counts().to_dict()}")

def safe_pattern(text: str) -> str:
    """Escape special regex characters for safe use in assertions."""
    return re.escape(str(text).strip())

Loaded 125 rows
Difficulty:    {'medium': 50, 'easy': 38, 'hard': 37}
Category:      {'logic': 39, 'math': 26, 'riddle': 23, 'language': 21, 'lateral': 16}
EF Component:  {'inhibitory_control_logical': 39, 'inhibitory_control_numerical': 26, 'inhibitory_control_semantic': 23, 'inhibitory_control_linguistic': 21, 'inhibitory_control_schematic': 16}
Scoring Method:{'exact_or_equivalent': 125}


In [2]:
from kaggle_benchmarks import llms
import kaggle_benchmarks as kbench

llm1 = kbench.llm   # Gemini Flash (baseline)

llm2 = llms.get("google/gemma-4-26b-a4b")   # weak

llm3 = llms.get("openai/gpt-5.4-mini-2026-03-17")   # mid

llm4 = llms.get("google/gemini-3.1-pro-preview")    # strong

llm5 = llms.get("anthropic/claude-sonnet-4-6@default")   # very strong

llm6 = llms.get("deepseek-ai/deepseek-r1-0528")    # reasoning-heavy

all_models = [llm1, llm2, llm3, llm4, llm5, llm6]

for i, m in enumerate(all_models, 1):
    status = "Loaded" if m else "Failed"
    print(f"llm{i}: {m} --> {status}")


llm1: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded
llm2: 🤖 google/gemma-4-26b-a4b --> Loaded
llm3: 🤖 openai/gpt-5.4-mini-2026-03-17 --> Loaded
llm4: 🤖 google/gemini-3.1-pro-preview --> Loaded
llm5: 🤖 anthropic/claude-sonnet-4-6@default --> Loaded
llm6: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded


In [3]:
# ── Task definition ───────────────────────────────────────────────────────────
@kbench.task(name="impulse_block", version=1)
def impulse_block(
    llm,
    id: int,
    question: str,
    tempting_wrong_answer: str,
    correct_answer: str,
    category: str,
    difficulty: str,
    explanation: str,
    answer_format: str,
    scoring_method: str,
    ef_component: str,
) -> tuple[int, int]:
    """Inhibitory control: resist the tempting wrong answer and produce the correct one. Categories: math, logic, language, riddle, lateral. Difficulty: easy/medium/hard."""

    response = llm.prompt(
        "Answer the following question carefully. "
        "The intuitive first answer is often wrong — think before you respond. "
        "Reply with ONLY the final answer, nothing else.\n\n"
        f"Question: {question}"
    )

    passes = 0

    # Assertion 1: correct answer must be present (scoring_method: exact_or_equivalent)
    r1 = kbench.assertions.assert_contains_regex(
        rf"(?i){safe_pattern(correct_answer)}",
        response,
        expectation=f"Must contain correct answer: '{correct_answer}' (ef_component: {ef_component})",
    )
    if r1.passed:
        passes += 1

    # Assertion 2: tempting wrong answer must NOT appear
    r2 = kbench.assertions.assert_not_contains_regex(
        rf"(?i)\b{safe_pattern(tempting_wrong_answer)}\b",
        response,
        expectation=(
            f"Must NOT contain tempting wrong answer: '{tempting_wrong_answer}'. "
            f"Correct: '{correct_answer}'. Explanation: {explanation}"
        ),
    )
    if r2.passed:
        passes += 1

    return passes, 2

In [4]:
# ── Smoke test ────────────────────────────────────────────────────────────────
print("\n── Smoke test (row 0) ──")
smoke = df.iloc[0]
run = impulse_block.run(
    llm=kbench.llm,
    id=int(smoke["id"]),
    question=smoke["question"],
    tempting_wrong_answer=smoke["tempting_wrong_answer"],
    correct_answer=smoke["correct_answer"],
    category=smoke["category"],
    difficulty=smoke["difficulty"],
    explanation=smoke["explanation"],
    answer_format=smoke["answer_format"],
    scoring_method=smoke["scoring_method"],
    ef_component=smoke["ef_component"],
)
print(f"Result: {run.result}  |  Passed: {run.passed}")
print("Smoke test complete ")


── Smoke test (row 0) ──


Result: (2, 2)  |  Passed: True
Smoke test complete 


In [5]:
# ── Multi-model evaluation ────────────────────────────────────────────────────
print("\n── Multi-model evaluation ──")
runs = impulse_block.evaluate(
    llm=all_models,
    evaluation_data=df,
    n_jobs=4,
    max_attempts=3,
    retry_delay=5,
)


── Multi-model evaluation ──


In [6]:
# ── Results ───────────────────────────────────────────────────────────────

results_df = runs.as_dataframe()

# Convert (passes, total) → score
results_df["score"] = results_df["result"].apply(
    lambda x: x[0] / x[1] if isinstance(x, tuple) and x[1] > 0 else float(x)
)

# Clean model names
results_df["model_name"] = results_df["llm"].apply(lambda x: str(x))

# ── BASIC STATS ───────────────────────────────────────────────────────────

print(f"\nTotal runs    : {len(results_df)}")
print(f"Overall score : {results_df['score'].mean():.3f}")

# ── BREAKDOWN ─────────────────────────────────────────────────────────────

print("\nScore by difficulty:")
print(results_df.groupby("difficulty")["score"].mean().round(3))

print("\nScore by category:")
print(results_df.groupby("category")["score"].mean().round(3))

# Only if column exists
if "ef_component" in results_df.columns:
    print("\nScore by ef_component:")
    print(results_df.groupby("ef_component")["score"].mean().round(3))

print("\nScore by model:")
print(results_df.groupby("model_name")["score"].mean().round(3))

# ── MODEL COMPARISON TABLE───────────────────

print("\n── Model comparison (table) ──")

pivot_df = results_df.pivot_table(
    index="id",
    columns="model_name",
    values="score"
)

print(pivot_df.round(3))


results_df.to_csv("final_results.csv", index=False)
pivot_df.to_csv("model_comparison.csv")

print("\nResults saved as CSV files")


Total runs    : 750
Overall score : 0.638

Score by difficulty:
difficulty
easy      0.660
hard      0.565
medium    0.675
Name: score, dtype: float64

Score by category:
category
language    0.563
lateral     0.516
logic       0.585
math        0.724
riddle      0.783
Name: score, dtype: float64

Score by ef_component:
ef_component
inhibitory_control_linguistic    0.563
inhibitory_control_logical       0.585
inhibitory_control_numerical     0.724
inhibitory_control_schematic     0.516
inhibitory_control_semantic      0.783
Name: score, dtype: float64

Score by model:
model_name
🤖 anthropic/claude-sonnet-4-6@default    0.676
🤖 deepseek-ai/deepseek-r1-0528           0.566
🤖 google/gemini-3.1-pro-preview          0.656
🤖 google/gemma-4-26b-a4b                 0.688
🤖 openai/gpt-5.4-mini-2026-03-17         0.676
Name: score, dtype: float64

── Model comparison (table) ──
model_name  🤖 anthropic/claude-sonnet-4-6@default  \
id                                                  
0           